# Post-processing and aggregation of detections

In [ ]:
import json
import os

import geopandas as gpd
import numpy as np
import shapely.geometry as sg

from zwerfafval_detectie.utils_eval import read_annotations_folder

RD_CRS = "EPSG:28992"  # CRS code for the Dutch Rijksdriehoek coordinate system
LAT_LON_CRS = "EPSG:4326"  # CRS code for WGS84 latitude/longitude coordinate system

In [ ]:
predictions_folder = "../datasets/experiments/zwerfafval/predict/inwinning_260713_26m_v1e2"
metadata_json_folder = "../datasets/experiments/zwerfafval/data_inwinning_260713/json_tasks"

categories = {
    0: "Zwerfafval (grof)",
    1: "Zwerfafval (fijn)"
}

confidence = 0.3

In [ ]:
# Load model predictions

predictions_gdf = read_annotations_folder(folder_path=predictions_folder, categories=categories)
predictions_gdf["file_name"] = predictions_gdf["file_name"].apply(lambda f: os.path.splitext(f)[0])

_predictions_sorted = (
    predictions_gdf[predictions_gdf["confidence"] >= confidence]
    .set_index("file_name")
    .sort_index()
)


# Count predictions per image

counts_df = (
    _predictions_sorted[["category"]]
    .replace(categories)
    .groupby(["file_name", "category"])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
# Load metadata from JSON files

data = {
    "file_name": [],
    "geometry": [],
}

metadata_files = sorted([
    file for file in os.listdir(metadata_json_folder) 
    if os.path.splitext(file)[1] == ".json"
])

for file in metadata_files:
    with open(os.path.join(metadata_json_folder, file), 'r') as fh:
        json_content = json.load(fh)
        data["file_name"].append(
            os.path.splitext(json_content["image_file_name"])[0]
        )
        data["geometry"].append(
            sg.Point((
                json_content["gps_data"]["longitude"],
                json_content["gps_data"]["latitude"]
            ))
        )

metadata_gdf = gpd.GeoDataFrame(
    data=data,
    crs=LAT_LON_CRS
).set_index("file_name")

In [ ]:
# Merge object counts and metadata

counts_merged = (
    gpd.GeoDataFrame(counts_df.join(metadata_gdf, how="outer"))
    .fillna(value={
        "Zwerfafval (fijn)": 0,
        "Zwerfafval (grof)": 0,
    })
    .to_crs(RD_CRS)
)

In [ ]:
# Select image if distance to previous selected image is larger than a threshold

min_distance = 5.0

points = counts_merged["geometry"].to_crs(RD_CRS)

previous = points.iloc[0]

selection = [False]*len(points)
selection[0] = True

for i, point in enumerate(points.iloc[1:]):
    if previous.distance(point) >= min_distance:
        previous = point
        selection[i+1] = True

counts_merged["selected"] = False
counts_merged.loc[:, "selected"] = selection

In [ ]:
# Compute average counts for each selected image by averaging all upcoming
# images until the next selected 

# The assumption is that the camera looks forward
# so the upcoming images are a good representation for the current situation

counts_merged.reset_index(inplace=True)
idx_selected = counts_merged.index[counts_merged["selected"]].tolist()

counts_merged["section_mean"] = np.nan

for i in range(0, len(idx_selected) - 1):
    start_idx = idx_selected[i]
    end_idx = idx_selected[i+1] - 1
    section_mean = counts_merged.loc[start_idx:end_idx, "Zwerfafval (grof)"].mean()
    counts_merged.loc[start_idx, "section_mean"] = section_mean

counts_merged = (
    counts_merged.loc[counts_merged["selected"], ["file_name", "section_mean", "geometry"]]
    .drop(index=counts_merged.index[-1])
    .set_index("file_name")
)

In [ ]:
# Compute rolling average over section counts
# A window of 5 means averaging over 25m stretches (since each section is 5m)

counts_merged["rolling_avg"] = counts_merged["section_mean"].rolling(window=5, min_periods=1, center=True).mean()

In [ ]:
# Plot results on a map

from xyzservices import TileProvider

ams_tile_provider = TileProvider(
    name="Topografie, standaard visualisatie (WM)",
    url="https://t1.data.amsterdam.nl/topo_wm/{z}/{x}/{y}.png",
    attribution="data.amsterdam.nl",
)

plot_column = "rolling_avg"
# plot_column = "section_mean"

map = (
    counts_merged
    .explore(
        column=plot_column,
        cmap="YlOrRd",
        style_kwds={
            "style_function": lambda x: {"radius": 2*x["properties"][plot_column]},
            "fillOpacity": 0.75,
            "weight": 2
        },
        legend=True,
        tiles=ams_tile_provider
    )
)

map.save(os.path.join("../datasets/experiments/zwerfafval", f"heatmap_inwinning_260713_sampled_roll.html"))
# map.save(os.path.join("../datasets/experiments/zwerfafval", f"heatmap_inwinning_260713_sampled.html"))